In [1]:
import os
import random
import pandas as pd
from tqdm import tqdm
import ast
import shutil
import git
import fnmatch

# Change it to your google drive path where this notebook located.
drive_path = '/Users/samyiin/Projects/ZipfLawAnalysis'
os.chdir(drive_path)
from Utils.EnglishDictionary import EnglishDictionary
from Utils.HardwordParser import HardwordParser

## Parse from Hard word to soft words (Cache Run)

In [2]:
df_names = pd.read_csv("Database/TempData/DataProcessing/df_hardwords.csv")
df_names['name'] = df_names['name'].astype(str)
hard_word_parser = HardwordParser()
df_names['softwords'] = df_names['name'].apply(hard_word_parser.parse_hard_word)


df_softwords = df_names.copy()
df_softwords['every_softword'] = df_softwords['softwords'].apply(lambda my_list: [s for s in my_list if not s.isdigit()])
# get the standardized names (with or without numbers
df_softwords['standard_name_nodigit'] = df_softwords['softwords'].apply(lambda my_list: "_".join(s.lower() for s in my_list if not s.isdigit()))
df_softwords['standard_name_digit'] = df_softwords['softwords'].apply(lambda my_list: "_".join(s.lower() for s in my_list))

# explode: make each softword a new row
df_softwords = df_softwords.explode('every_softword')

df = df_softwords
non_strings = df[~df['every_softword'].apply(lambda x: isinstance(x, str))]
unique_names = non_strings['name'].unique()
number_nonstrings = len(non_strings)
print(f"There are {number_nonstrings} names that's without english letter, they are {unique_names} so their softword are nan, softword list are []")

df_softwords.to_csv("Database/TempData/DataProcessing/df_softwords.csv", index=False)

There are 1437 names that's without english letter, they are ['*' '_' '__' '_2011'] so their softword are nan, softword list are []


## Parse softwords (Cache Run)

In [3]:
# load the heuristic parser
from Utils.HeuristicSoftwordParser import HeuristicParser
heruristic_parser = HeuristicParser()
df_softwords = pd.read_csv("Database/TempData/DataProcessing/df_softwords.csv", keep_default_na=False)


In [4]:
# add the catagory, explanation and suggestions to df_terms

def classify_term(row):
    softword = row["every_softword"]
    if not isinstance(softword, str):
        return {"temp_col": 0, "interpretation": 0} # names like ['*' '_' '__' '_2011'] has nan as softword
        
    interpretation = heruristic_parser.parse(softword)
    if interpretation:
        return {"temp_col": 1, "interpretation": interpretation}
    return {"temp_col": -1, "interpretation": -1}

df = df_softwords.copy()

# Apply the classification function row-wise
df = pd.concat(
    [
        df,
        df.apply(classify_term, axis=1, result_type="expand")  # Expands the dictionary into new columns
    ],
    axis=1
)
df = df.reset_index(level=None, drop=True, inplace=False)

In [5]:
# !pip install openai==0.28
import pandas as pd
import openai
import json
from Utils.Secrete import SecreteLoader
from tqdm import tqdm
from Utils.SemanticSoftwordParser import SemanticSoftwordParser

def sementic_parse(softword, standard_name):
    # save the file under the directory as a "cache"
    filename = softword + "_" + standard_name + ".json"
    cache_directory = "Database/TempData/DataProcessing/unidentifiable_mapping_cache/"
    filepath = os.path.join(cache_directory, filename)

    # if already exist then don't query again
    if os.path.exists(filepath):
        with open(filepath, "r", encoding="utf-8") as f:
            stored_info = json.load(f)
        return stored_info['response']

    # query gpt: get the json format we want.
    try:
        my_openai_apikey = SecreteLoader().get_openai_api_key("Sam")
        semantic_parser = SemanticSoftwordParser(my_openai_apikey)
        soft_word_context = f'This word is used in the Python programming identifier name "{standard_name}".'
        response = semantic_parser.parse(softword, soft_word_context)
    except:
        # we already know this is not happening. 
        print(softword, soft_word_context)
        return None

    # save the response
    stored_info = {"reasoning_process": semantic_parser.get_reasoning_process(),
                   "response": response
                  }
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(stored_info, f, indent=2, ensure_ascii=False)

    return stored_info['response']
        
# running a for loop once for the cache, that's faster.
df = df[df['interpretation'] == -1]
# should get rid off import alias and stuff?
# should pass in original name?
df["every_softword"] = df["every_softword"].str.lower()
unique_pairs = df.groupby(['standard_name_nodigit', 'every_softword']).size().reset_index(name='count').sort_values(by="count", ascending=False)

# These are the ones we will pass to LLM, let's see the top 50
unique_pairs.to_csv("Database/TempData/DataProcessing/unique_non_dictionary.csv")
unique_pairs

df = unique_pairs
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing softwords"):
    softword = row["every_softword"]
    standard_name = row["standard_name_nodigit"]

    sementic_parse(softword, standard_name)

Processing softwords: 100%|███████████████████████████████████████████████████████████| 7473/7473 [00:01<00:00, 6865.95it/s]


In [6]:
def create_softword_interpretation_map(softword, standard_name):
    # try the heuristic parser
    interpretation = heruristic_parser.parse(softword)
    if interpretation:
        return {"interpreter": "heuristic", "interpretation": interpretation['interpretation']['split']}
        
    # if not then try the semantic parser
    interpretation = sementic_parse(softword, standard_name)
    if interpretation:
        return {"interpreter": "semantic", "interpretation": interpretation['interpretation']['split']}
    
    # probably won't get to this cell 
    return {"interpreter": None, "interpretation": None}

my_dictionary = {}
for _, row in tqdm(df_softwords.iterrows(), total=len(df_softwords), desc="Creating dictionary:"):
    softword = row["every_softword"]
    if not isinstance(softword, str):
        continue    # no need create mapping for it
    softword = softword.lower()
    standard_name = row["standard_name_nodigit"]
    my_dictionary[(softword, standard_name)] = create_softword_interpretation_map(softword, standard_name)
    
len(my_dictionary)

Creating dictionary:: 100%|██████████████████████████████████████████████████████| 583285/583285 [00:08<00:00, 65126.36it/s]


156139

## Parse Hardwords (Actual Run)

In [7]:
df_names = pd.read_csv("Database/TempData/DataProcessing/df_hardwords.csv")
df_names['name'] = df_names['name'].astype(str)
hard_word_parser = HardwordParser()
df_names['softwords'] = df_names['name'].apply(hard_word_parser.parse_hard_word)
df_names['softwords_nodigit'] = df_names['softwords'].apply(lambda my_list: [s for s in my_list if not s.isdigit()])
# get the standardized names (with or without numbers
df_names['standard_name_nodigit'] = df_names['softwords'].apply(lambda my_list: "_".join(s.lower() for s in my_list if not s.isdigit()))
df_names['standard_name_digit'] = df_names['softwords'].apply(lambda my_list: "_".join(s.lower() for s in my_list))

def interpret_hardword(row):
    hardword_interpretation = []
    for softword in row['softwords_nodigit']:
        hardword_interpretation.extend(my_dictionary[(softword.lower(), row['standard_name_nodigit'])]['interpretation'])
    return hardword_interpretation


tqdm.pandas()

df_names['hardword_interpretation'] = df_names.progress_apply(interpret_hardword, axis=1)

100%|███████████████████████████████████████████████████████████████████████████| 406842/406842 [00:01<00:00, 213222.71it/s]


In [8]:
df_users = pd.read_csv('Database/TempData/GatherData/Filter_4_SendUserEmail/Results/all_users.csv')
df_names = pd.merge(
    df_names,
    df_users[['login', 'claimed_country', 'native_language', 'parent_native_language']],
    on='login',
    how='left'  # Use 'left' join to keep all rows in df1
)

# rename the columns incase collision
df_names = df_names.rename(columns={"login": "user_login"})

df_names.to_csv("Database/TempData/DataProcessing/df_names_sensitive.csv", index=False)